# Credit Card Fraud Detection Project
This project analyzes credit card transactions to detect potentially fraudulent activities using Machine Learning.

### **Project Goals:**
1.   **EDA**: Analyze the frequency and characteristics of fraudulent transactions.
2.   **Prediction**: Predict whether a transaction is fraudulent or legitimate based on historical data.

### **1. Setup and Library Imports**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, auc
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterignore("ignore")

sns.set_style("whitegrid")

AttributeError: module 'warnings' has no attribute 'filterignore'

### **2. Loading the Dataset**
**Note**: You can download `creditcard.csv` from [Kaggle](https://www.kaggle.com/mlg-ulb/creditcardfraud). If using Google Colab, upload the file manually or use the Kaggle API.

In [ ]:
# Loading the dataset
try:
    df = pd.read_csv('creditcard.csv')
    print("Dataset loaded successfully with shape:", df.shape)
except FileNotFoundError:
    print("File not found. Please upload 'creditcard.csv' to the project directory.")

### **3. Explortory Data Analysis (EDA)**
The user specifically asked for an analysis of how many frauds are in the data.

In [ ]:
# Inspect the distribution of classes (Fraud vs Normal)
class_counts = df['Class'].value_counts()
fraud_percentage = (class_counts[1] / len(df)) * 100

print(f"Number of Normal transactions: {class_counts[0]}")
print(f"Number of Fraudulent transactions: {class_counts[1]}")
print(f"Fraud Percentage: {fraud_percentage:.3f}%")

In [ ]:
# Visualizing the distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='Class', data=df, palette='viridis')
plt.title('Fraud vs Normal Transaction Distribution')
plt.xlabel('Class (0: Normal, 1: Fraud)')
plt.ylabel('Count')
plt.show()

### **4. Data Preprocessing**
We need to scale the `Time` and `Amount` features as others (V1-V28) are already PCA transformed.

In [ ]:
scaler = StandardScaler()
df['Scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['Scaled_Time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Removing old features
df.drop(['Time', 'Amount'], axis=1, inplace=True)

### **5. Handling Data Imbalance**
Since fraudulent entries represent only ~0.17% of total transactions, we will use **SMOTE** to over-sample the minority class (Fraud).

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Splitting into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Resampled training set count:", len(y_train_resampled))

### **6. Building the Machine Learning Model**
We'll use a **Random Forest Classifier** for high accuracy.

In [ ]:
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train_resampled, y_train_resampled)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

### **7. Model Evaluation**
We'll evaluate using Precision, Recall, Confusion Matrix, and ROC-AUC Score.

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### **8. Predicting Future Outcomes**
Create a function to predict fraud on new data.

In [ ]:
def predict_fraud(transaction_data):
    """
    Predicts if a single transaction (as a numpy array/list) is fraud or not.
    """
    prediction = model.predict([transaction_data])
    return "Fraud" if prediction[0] == 1 else "Normal"

# Example usage:
sample_transaction = X_test.iloc[0].values
result = predict_fraud(sample_transaction)
print(f"Outcome for Sample Transaction: {result}")

NameError: name 'X_test' is not defined